In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt
import prism
import pandas as pd
from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.buildings.preprocessing.floorspace import get_image_floorspace, get_floorspace_urban_rural
from imagematerials.buildings.preprocessing.population import compute_population

In [ ]:
scenario_list = {
                "base":("SSP2",["base"]),
                # "narrow":("SSP2_narrow", ["base", "narrow"]),
                # "slow":("SSP2_slow",["base", "slow"]),
                # "close":("SSP2",["base", "close"]),
                # "narrow_slow_close":("SSP2_narrow_slow_close", ["base", "narrow","slow", "close"])
                 }

In [ ]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1920
complete_timeline = prism.Timeline(time_start, 2100, 1)
simulation_timeline = prism.Timeline(1970, 2100, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dirs = {
        scenario: scenario_base_path / scenario for scenario in circular_scen
    }

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs) 
    vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs)

    #TODO fix this for real in the future
    prep_data = vhc_sector.prep_data

    target_materials = [
    "Aluminium", 
    "Brick", 
    "Cement", 
    "Concrete", 
    "Copper", 
    "Glass", 
    "Steel", 
    "Wood"
    ]

    prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['battery_materials'] = prep_data['battery_materials'].reindex(material=target_materials)
    prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['material_fractions'] = prep_data['material_fractions'].reindex(material=target_materials)
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].reindex(material=target_materials)

    vhc_sector = Sector('vehicles', prep_data)

    prep_eol = eol_preprocess(Path("..", "data", "raw"), circular_economy_scenario_dirs)
    eol_sector = Sector(name="eol", data = prep_eol)

    factory = ModelFactory(
    [bld_sector, 
    vhc_sector, 
     eol_sector], complete_timeline
    ).add(GenericStocks, ["buildings", 
                         "vehicles"
                          ]
    ).add(GenericMaterials,  "vehicles"
    ).add(Maintenance, "vehicles"
    ).add(MaterialIntensities, "buildings",
    ).add(EndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": [
        "buildings",
        "vehicles", 
        ],
    "inflow_materials": [
        "buildings",
        "vehicles", 
        ],
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
    )
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

        # simple output to be used in TIMER (for inflows, reusable and recyclable steel and concrete only)
    all_output[scen_id] = {

        "inflow_materials": [
            model.vehicles["inflow_materials"], 
            model.buildings["inflow_materials"]
                             ],
       "stock_by_cohort_materials": [
           model.vehicles["stock_by_cohort_materials"], 
           model.buildings["stock_by_cohort_materials"]],
       "outflow_by_cohort_materials": 
       [
            model.vehicles["outflow_by_cohort_materials"], 
            model.buildings["outflow_by_cohort_materials"]],
        
        "inflow": [
            model.vehicles["inflow"], 
            model.buildings["inflow"]],

        "stocks": [
            model.vehicles["stocks"], 
            model.buildings["stocks"]],
        "outflow": [
            model.vehicles["stocks"], 
            model.buildings["stocks"]],
            
       "sum_outflow": model.eol["sum_outflow"],
       "sum_inflow": model.eol["sum_inflow"],
       "collected_materials": model.eol["collected_materials"],
       "reusable_materials": model.eol["reusable_materials"],
       "recyclable_materials": model.eol["recyclable_materials"],
       "losses_materials": model.eol["losses_materials"],
       "virgin_materials": model.eol["virgin_materials"]
    }
    print(f"Finished {scen_id}")

    
    # full output to be reported (all materials, all variables) 
    #TODO: add all variables to the output

    # all_output[scen_id] = {
        # "inflow_materials": [model.vehicles["inflow_materials"], model.buildings["inflow_materials"]],
        # "stocks": [model.vehicles["stocks"], model.buildings["stocks"]],
    #     "outflow_by_cohort_materials": [model.vehicles["outflow_by_cohort_materials"], model.buildings["outflow_by_cohort_materials"]],
    #    
    #     "reusable_materials": model.eol["reusable_materials"], 
    #     "recyclable_materials": model.eol["recyclable_materials"]
    # }
    # print(f"Finished {scen_id}")

In [ ]:
from imagematerials.reporting import create_iamc_reporting, create_iamc_eol
from imagematerials.reporting import iamc_config as CFG
from types import SimpleNamespace

DESIRED_SCENARIOS = [
    "SSP2", 
    #"SSP2_narrow",
    #"SSP2_slow",
    #"SSP2_close",
    #"SSP2_narrow_slow_close",
    # "SSP2_19",
    # "SSP2_narrow_19",
    # "SSP2_slow_19",
    # "SSP2_close_19",
    # "SSP2_narrow_slow_close_19",
    # "SSP2_26",
    # "SSP2_narrow_slow_close_26",
    ]  

In [ ]:


output_vhc = {
    iamc_label: SimpleNamespace(
        vehicles={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][0],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][0],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][0],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][0],
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][0],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][0],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

# Vehicles reporting
df_vhc = create_iamc_reporting(
    models=output_vhc,
    templates=[
        "Final Product Demand|Transportation|{Vehicles}",
       "Product Stock|Transportation|{Vehicles}",
       "Stock Retirement|Transportation|{Vehicles}",
        "Final Material Demand|{Engineered Material}|{Demand Sector}",
       "Material Stock|{Engineered Material}|{Demand Sector}",
       "Material Outflow|{Engineered Material}|{Demand Sector}",
    ],
    sector="vehicles",               
    model_name="IMAGE 3.4",
    outfile="vehicles.xlsx",
)


output_bld = {
    iamc_label: SimpleNamespace(
        buildings={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][1],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][1],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][1],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][1],
            "stock_by_cohort":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],  # Map stocks to stock_by_cohort
            "outflow_by_cohort": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],  # Map outflow to outflow_by_cohort
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}


# Buildings reporting
df_bld = create_iamc_reporting(
    models = output_bld,
    templates=[
        "Final Product Demand|Buildings|{Building Types}",
        "Product Stock|Buildings|{Building Types}",
        "Stock Retirement|Buildings|{Building Types}",
        "Final Material Demand|{Engineered Material}|{Demand Sector}",
        "Material Stock|{Engineered Material}|{Demand Sector}",
        "Material Outflow|{Engineered Material}|{Demand Sector}",
    ],
    sector = "buildings",
    model_name="IMAGE 3.4",
    outfile="buildings.xlsx",
)

df_combined = pd.concat([df_vhc, df_bld], ignore_index=True)
df_combined.to_excel("reporting_materials.xlsx", index=False)


In [ ]:
import warnings
from pint.errors import UnitStrippedWarning

output_eol = {
    iamc_label: SimpleNamespace(
        eol={
            "sum_outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["sum_outflow"],
            "collected_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["collected_materials"],
            "reusable_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["reusable_materials"],
            "recyclable_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["recyclable_materials"],
            "losses_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["losses_materials"],
            "virgin_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["virgin_materials"],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}



# End-of-life reporting
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UnitStrippedWarning)

    df_eol = create_iamc_eol(
        models = output_eol,
        templates=[
            "Material Losses|{Engineered Material}",
            "Final Material Demand|{Engineered Material}|{Demand Sector}|Reused",
            "Final Material Demand|{Engineered Material}|{Demand Sector}|Recycled",
            "Final Material Demand|{Engineered Material}|{Demand Sector}|Virgin Input",
            "Scrap|Iron and Steel|Steel|{Demand Sector}",
            "Scrap|Non-Ferrous Metals|Aluminum|{Demand Sector}",
            "Scrap|Iron and Steel|Steel",
            "Scrap|Non-Ferrous Metals|Aluminum"

        ],
        sector = "eol",
        model_name="IMAGE 3.4",
        outfile="eol.xlsx",
    )


In [ ]:
import warnings
import pandas as pd
from pint.errors import UnitStrippedWarning

warnings.filterwarnings("ignore", category=UnitStrippedWarning)

output_dir = Path("..","data","output","TIMER_input")
output_dir.mkdir(exist_ok=True)

materials = ["Steel", "Concrete"]
common_vars = ["inflow_materials", "outflow_by_cohort_materials"]
eol_vars = ["sum_outflow", "reusable_materials", "recyclable_materials"]
variable_units = {
    "inflow_materials": "Mt",
    "outflow_by_cohort_materials": "Mt",
    "reusable_materials": "Mt",
    "recyclable_materials": "Mt"
}
sector_unit_conversion = {
    "vehicles": {"factor": 1e-9, "unit": "Mt"},             # kg to Mt
    "buildings": {"factor": 1e-9, "unit": "Mt"},            # kg to Mt
    "eol": {"factor": 1e-9, "unit": "Mt"}               # kg to Mt       
                                                            
}

all_rows = []


for scen_id, outputs in all_output.items():
    print(f"Processing {scen_id}")
    rows = []

    for mat in materials:
        # vehicles and buildings
        for var in common_vars:
            for i, sec in enumerate(["vehicles", "buildings"]):
                da = outputs[var][i].to_array().sel(material=mat).sum(dim="Type")
                df = da.to_dataframe(name="value").reset_index()
                # unit conversion
                conv = sector_unit_conversion[sec]
                df["value"] = df["value"] * conv["factor"]
                df["unit"] = conv["unit"]
                df["material"] = mat
                df["sector"] = sec
                df["variable"] = var
                df["scenario"] = scen_id
                rows.append(df)

        # eol 
        for var in eol_vars:
            da = outputs[var].to_array().sel(material=mat).sum(dim="Type")
            df = da.to_dataframe(name="value").reset_index()
            df["value"] = df["value"] * sector_unit_conversion["eol"]["factor"]
            df["unit"] = sector_unit_conversion["eol"]["unit"]
            df["material"] = mat
            df["sector"] = "eol"
            df["variable"] = var
            df["scenario"] = scen_id
            rows.append(df)

    all_rows.extend(rows)
    print(f"Finished {scen_id}")

# Combine all rows into a single DataFrame and pivot it
full_df = pd.concat(all_rows, ignore_index=True)
pivot_df = full_df.pivot_table(
    index=["scenario", "Region", "material", "variable", "unit", "sector"],
    columns="time",  
    values="value"
).reset_index()

pivot_df.to_csv(output_dir / "IMAGE_materials_to_TIMER.csv", index=False)